In [ ]:
!pip install docling
!pip install -U transformers
!pip install qwen-vl-utils
!pip install pymupdf

In [ ]:
# Aufteilen des Modulhandbuchs in einzelne Seiten
import fitz  # PyMuPDF

def pdf_in_einzelseiten(input_pdf, output_prefix="Seite_"):
    doc = fitz.open(input_pdf)
    
    for i, seite in enumerate(doc, start=1):
        # Neues Dokument für jede Seite
        neue_pdf = fitz.open()
        neue_pdf.insert_pdf(doc, from_page=i-1, to_page=i-1)
        
        # Speichern
        neue_pdf.save(f"{output_prefix}{i}.pdf")
        neue_pdf.close()
    
    doc.close()

# Beispielaufruf
pdf_in_einzelseiten("Modulhandbuch.pdf", output_prefix="pdfs/Seite_")

In [24]:
# 2 Seiten nebeneinander kleben
import fitz

# DIN A4 Querformat Maße (Breite > Höhe)
a4_width, a4_height = fitz.paper_size("a4")
# Für Querformat Höhe und Breite tauschen
a4_width, a4_height = a4_height, a4_width

doc = fitz.open("Modul.pdf")
new_doc = fitz.open()

page1 = doc.load_page(0)
page2 = doc.load_page(1)

new_page = new_doc.new_page(width=a4_width, height=a4_height)

# Seitenrechtecke
r1 = page1.rect
r2 = page2.rect

# Breite und Höhe der einzelnen Seiten
w1, h1 = r1.width, r1.height
w2, h2 = r2.width, r2.height

# Skalierungsfaktor, damit beide Seiten nebeneinander auf A4 quer passen
scale = min((a4_width / 2) / max(w1, w2), a4_height / max(h1, h2))

scaled_w1, scaled_h1 = w1 * scale, h1 * scale
scaled_w2, scaled_h2 = w2 * scale, h2 * scale

# Zielrechtecke für die zwei Seiten
target_rect1 = fitz.Rect(0, 0, scaled_w1, scaled_h1)
target_rect2 = fitz.Rect(scaled_w1, 0, scaled_w1 + scaled_w2, scaled_h2)

# Seiten ohne Rotation einfügen, nur skaliert und nebeneinander
new_page.show_pdf_page(target_rect1, doc, 0)
new_page.show_pdf_page(target_rect2, doc, 1)

# Ergebnis speichern
new_doc.save("output_two_pages_a4_landscape.pdf")

In [29]:
# 2 Seiten untereinander kleben
import fitz

# DIN A4 Hochformat Maße
a4_width, a4_height = fitz.paper_size("a4")

doc = fitz.open("Modul.pdf")
new_doc = fitz.open()

page1 = doc.load_page(0)
page2 = doc.load_page(1)

new_page = new_doc.new_page(width=a4_width, height=a4_height)

# Rechtecke der Originalseiten
r1 = page1.rect
r2 = page2.rect

w1, h1 = r1.width, r1.height
w2, h2 = r2.width, r2.height

# Skalierungsfaktor, damit zwei Seiten untereinander auf A4 passen
scale = min(a4_width / max(w1, w2), (a4_height / 2) / max(h1, h2))

scaled_w1, scaled_h1 = w1 * scale, h1 * scale
scaled_w2, scaled_h2 = w2 * scale, h2 * scale

# Zielrechtecke für ober und untere Seite
target_rect1 = fitz.Rect(0, 0, scaled_w1, scaled_h1)
target_rect2 = fitz.Rect(0, scaled_h1, scaled_w2, scaled_h1 + scaled_h2)

# Seiten einfügen ohne Rotation
new_page.show_pdf_page(target_rect1, doc, 0)
new_page.show_pdf_page(target_rect2, doc, 1)

# Resultat speichern
new_doc.save("output_two_pages_stacked.pdf")

In [30]:
from docling.datamodel.base_models import InputFormat
from docling.document_extractor import DocumentExtractor

extractor = DocumentExtractor(allowed_formats=[InputFormat.PDF])

result = extractor.extract(source = "output_two_pages_stacked.pdf", template = {

    "Modul": "String",
    "Workload": "String", 
    "Credits nach ECTS": "float", 
    "Studiensemester": "String",
    "Häufigkeit des Angebots": "String",
    "Dauer": "String",
    "Lehrveranstaltungen": "String",
    "Kontaktzeit": "String",
    "Selbststudium": "String",
    "geplante Gruppengröße": "String",
    "Lernergebnisse": "String",
    "Inhalte": "String",
    "Lehr- und Lernformen": "String",
    "Teilnahmevoraussetzungen": "String",
    "Prüfungsformen": "String",
    "Prüfungsvorleistung": "String",
    "Voraussetzungen für die Vergabe von Kreditpunkten": "String",
    "Verwendung des Moduls": "String",
    "Stellenwert der Note für die Endnote": "String",
    "Modulbeauftragte*r und hauptamtlich Lehrende": "String",
    "Sonstige Informationen": "String",
})

In [32]:
print(result.pages[0].extracted_data)

{'Modul': 'Kryptographie', 'Workload': '150 h', 'Credits nach ECTS': 6, 'Studiensemester': '1. Semester', 'Häufigkeit des Angebots': 'Wintersemester', 'Dauer': '1 Semester', 'Lehrveranstaltungen': '2 SWS Vorlesung & 1 SWS Übung (als Lehrbrief) 1 SWS Übung', 'Kontaktzeit': '25 h', 'Selbststudium': '125 h', 'geplante Gruppengröße': '30 Studierende', 'Lernergebnisse': 'Die Studierenden kennen die Bedeutung der Schutzziele Datenintegrität, Vertraulichkeit, Authentizität und Verbindlichkeit sowie Methoden und kryptographische Primitiven, die zur Erreichung dieser Schutzziele eingesetzt werden. Sie verstehen die grundlegenden Sicherheitsbegriffe der Kryptographie und können in einfachen Szenarien beurteilen, mit welchen kryptographischen Verfahren und Protokollen nach aktuellem Stand der Technik die Schutzziele erreicht werden können.', 'Inhalte': 'Grundlagen, Schutzziele der IT-Sicherheit, Angriffsszenarien und Sicherheitsbegriffe, Kryptographische Primitiven, historische Verschlüsselungsve

In [ ]:
from collections import defaultdict

test_dict = {}
none_keys = []
page_num = 0

studiengaenge = defaultdict(dict)

# Iterieren über alle Seiten der Ergebnisse, Ergebnisse der ersten Seite in dict speichern und Keys merken, die nicht auf der ersten Seite zu finden waren.
# Danach auf der zweiten Seite nur die Keys verwenden, die auf der ersten seite nicht gefunden wurden

for page in result.model_dump()["pages"]:
    data = page["extracted_data"]

    for key, value in data.items():
        if value and page_num == 0:
            test_dict[key] = value
            
        elif page_num == 1 and key in none_keys:
            test_dict[key] = value
        
        else:
            none_keys.append(key)
            
    page_num = page_num + 1
    
modulname = test_dict.pop("Modul")
studiengaenge["Angewandte Informatik"] [modulname] = test_dict

print(studiengaenge)